In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

## I can now load directly from src via my .toml file.
from src.data import remove_zero_variance, load_data


## Get data
X_df, y_df = load_data()



Removing 267 zero-variance genes.


Lets test logistic regression first for classification.

In [164]:
from sklearn.model_selection import train_test_split, GridSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

## Load the training data
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size = 0.2, stratify = y_df, random_state = 42)

## Set model
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

## Need to define hyperparameter grid for GridSearchCV

param_grid = {'classifier__C' : [0.001, 0.01, 0.1, 1.0, 10.0]}

logreg_grid_search =  GridSearchCV(
    estimator = pipeline, param_grid = param_grid, 
    cv = 5, scoring = 'accuracy', error_score="raise")


## Fit model
logreg_grid_search.fit(X_train, y_train)

## Get predictions
y_pred = logreg_grid_search.predict(X_test)

## Calculate and display accuracy metrics
logreg_accuracy = accuracy_score(y_test, y_pred)
logreg_report = classification_report(y_test, y_pred)
logreg_cv_accuracy = logreg_grid_search.best_score_

print(logreg_accuracy)
print("Best parameters:", logreg_grid_search.best_params_)
print("Best CV accuracy:", logreg_grid_search.best_score_)
print(logreg_report)

print("Best parameters:", logreg_grid_search.best_params_)
print("Best CV accuracy:", logreg_grid_search.best_score_)

#Best parameters: {'classifier__C': 0.001}
#Best CV accuracy: 0.9984375


0.9875776397515528
Best parameters: {'classifier__C': 0.001}
Best CV accuracy: 0.9984375
              precision    recall  f1-score   support

        BRCA       0.98      1.00      0.99        60
        COAD       1.00      0.94      0.97        16
        KIRC       1.00      1.00      1.00        30
        LUAD       0.96      0.96      0.96        28
        PRAD       1.00      1.00      1.00        27

    accuracy                           0.99       161
   macro avg       0.99      0.98      0.98       161
weighted avg       0.99      0.99      0.99       161

Best parameters: {'classifier__C': 0.001}
Best CV accuracy: 0.9984375


Wow logistic regression achieved 98.8% accuracy on test set. Now lets see if random forest does any better. It will at least let me pull out the genes that are most informative in the classification. 

In [165]:
from sklearn.ensemble import RandomForestClassifier

## Load the training data
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size = 0.2, stratify = y_df, random_state = 42)

## Initiate a pipeline. Scaling does not matter for randome forest 
## so I will leave it out of pipeline for now


rf_pipeline = Pipeline([
    ("classifier", RandomForestClassifier(random_state=42))
])

## Define param grid
param_grid = {
    "classifier__n_estimators": [100, 300, 500],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5],
}

rf_grid_search =  GridSearchCV(
    estimator = rf_pipeline, param_grid = param_grid, 
    cv = 5, scoring = 'accuracy', error_score="raise", n_jobs=-1)

## Fit model
rf_grid_search.fit(X_train, y_train)

## Get predictions
y_pred = rf_grid_search.predict(X_test)

## Calculate and display accuracy metrics
rf_accuracy = accuracy_score(y_test, y_pred)
rf_report = classification_report(y_test, y_pred)
rf_cv_accuracy = rf_grid_search.best_score_
print("Test: ", rf_accuracy)
print("Best parameters:", rf_grid_search.best_params_)
print("Best CV accuracy:", rf_grid_search.best_score_)
print(rf_report)



/Users/shanewarland/miniforge3/envs/cancer-ml/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Test:  0.9937888198757764
Best parameters: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 100}
Best CV accuracy: 0.996875
              precision    recall  f1-score   support

        BRCA       0.98      1.00      0.99        60
        COAD       1.00      1.00      1.00        16
        KIRC       1.00      1.00      1.00        30
        LUAD       1.00      0.96      0.98        28
        PRAD       1.00      1.00      1.00        27

    accuracy                           0.99       161
   macro avg       1.00      0.99      0.99       161
weighted avg       0.99      0.99      0.99       161



In [166]:
print(type(grid_search.best_estimator_))

<class 'sklearn.pipeline.Pipeline'>


Ok now lets try XGboost. Quite hard to imrpove 99.37% accuracy

In [167]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## Load the training data
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size = 0.2, stratify = y_df, random_state = 42)

## Initiate a pipeline. Scaling does not matter for randome forest 
## so I will leave it out of pipeline for now

xgb_pipeline = Pipeline([
    ("classifier", XGBClassifier(random_state=42))
])

## Define param grid
param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.1, 1],
}

xgb_grid_search =  GridSearchCV(
    estimator = xgb_pipeline, param_grid = param_grid, 
    cv = 5, scoring = 'accuracy', error_score="raise", n_jobs=-1)

## One last thing. XGBoost cannot use string labels
le = LabelEncoder()
y_numeric = le.fit_transform(y_train)

## Fit model
xgb_grid_search.fit(X_train, y_numeric)

## Get predictions
predictions = xgb_grid_search.predict(X_test)
y_pred = le.inverse_transform(predictions) ## undo the encoding

## Calculate and display accuracy metrics
xgb_accuracy = accuracy_score(y_test, y_pred)
xgb_report = classification_report(y_test, y_pred)
xgb_cv_accuracy = xgb_grid_search.best_score_
print("Test: ", xgb_accuracy)
print("Best parameters:", xgb_grid_search.best_params_)
print("Best CV accuracy:", xgb_grid_search.best_score_)
print(xgb_report)

/Users/shanewarland/miniforge3/envs/cancer-ml/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Test:  0.9813664596273292
Best parameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 50}
Best CV accuracy: 0.990625
              precision    recall  f1-score   support

        BRCA       0.95      1.00      0.98        60
        COAD       1.00      0.88      0.93        16
        KIRC       1.00      1.00      1.00        30
        LUAD       1.00      0.96      0.98        28
        PRAD       1.00      1.00      1.00        27

    accuracy                           0.98       161
   macro avg       0.99      0.97      0.98       161
weighted avg       0.98      0.98      0.98       161



In [170]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "CV Accuracy": [
        logreg_cv_accuracy,
        rf_cv_accuracy,
        xgb_cv_accuracy 
    ],
    "Test Accuracy": [
        logreg_accuracy,
        rf_accuracy,
        xgb_accuracy
    ]
})

print(results)

results.to_csv('../figures/results.csv', index=False)

                 Model  CV Accuracy  Test Accuracy
0  Logistic Regression     0.998437       0.987578
1        Random Forest     0.996875       0.993789
2              XGBoost     0.990625       0.981366


Ok Random Forest won. Although they were all so close. Logistic regression is much faster so that's worth considering. XGBoost took a while on my computer. Next we will dig into what the top performing model (Random Forest) tells us about the genes resposnible for the disease types. 

In [ ]:
## Save the different models
import joblib
joblib.dump(rf_grid_search.best_estimator_, '../models/rf_best_model.pkl')
joblib.dump(logreg_grid_search.best_estimator_, '../models/logreg_best_model.pkl')
joblib.dump(xgb_grid_search.best_estimator_, '../models/xgb_best_model.pkl')


['../models/xgb_best_model.pkl']

Next we move to 03_model_interpretation for analysis.